# Building a Report Generator


In [ ]:
%run "Setup_Env.ipynb"

## Branching and Merging Chains with LCEL

The idea here is to have multiple branching LLM Chains which work independently in parallel and then we merge their outputs finally using a merge LLM chain at the end to get a consolidated output

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
description_prompt =  ChatPromptTemplate.from_template(
    """Generate a two line description for the given topic:
      {topic}
""")

description_chain = (
    description_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [ ]:
pro_prompt = ChatPromptTemplate.from_template(
    """Generate three bullet points talking about the pros for the given topic:
      {topic}
""")

pro_chain = (
    pro_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [ ]:
con_prompt = ChatPromptTemplate.from_template(
    """Generate three bullet points talking about the cons for the given topic:
      {topic}
""")

con_chain = (
    con_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

branch_chain = (
    RunnableParallel(
        topic=itemgetter('topic'),
        description=description_chain,
        pros=pro_chain,
        cons=con_chain,
    )
)

In [ ]:
branch_chain = (
    RunnableParallel(
        topic=RunnablePassthrough(),
        description=description_chain,
        pros=pro_chain,
        cons=con_chain,
    )
)

In [ ]:
branch_chain.invoke({"topic": "Generative AI"})

In [ ]:
merge_prompt = ChatPromptTemplate.from_template(
    """Create a report about {topic} with the following information:
      Description:
      {description}
      Pros:
      {pros}
      Cons:
      {cons}

      Report should be in the following format:

      Topic: <name of the topic>

      Description: <description of the topic>

      Pros and Cons:

      <table with two columns showing the 3 pros and cons of the topic>
""")

merge_chain = (
    merge_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [ ]:
final_chain = (
    branch_chain
      |
    merge_chain
)

In [ ]:
response = final_chain.invoke({"topic": "Generative AI"})

In [ ]:
from IPython.display import Markdown, display

display(Markdown(response))